In [1]:
import os
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, utils

In [23]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
latent_dim = 100
batch_size = 4  # smaller batch to save memory
lr = 0.0002
epochs = 80
img_size = 128  # resize images to reduce memory
channels = 3

dataset_path = "./Dataset"  # update path to your dataset folder
output_dir = "./GAN_Output2"
model_dir = "./GAN_Model2"

os.makedirs(output_dir, exist_ok=True)
os.makedirs(model_dir, exist_ok=True)

In [24]:
transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*channels, [0.5]*channels)
])

class GANImageDataset(Dataset):
    def __init__(self, folder_path, transform=None):
        self.image_files = [
            os.path.join(folder_path, f)
            for f in os.listdir(folder_path)
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ]
        self.transform = transform

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        with Image.open(img_path) as image:
            image = image.convert('RGB')
            if self.transform:
                image = self.transform(image)
        return image

dataset = GANImageDataset(dataset_path, transform=transform)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=1)
print(f"Loaded {len(dataset)} images for GAN training.")

Loaded 1018 images for GAN training.


In [25]:
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(True),
            nn.Linear(256, 512),
            nn.ReLU(True),
            nn.Linear(512, 1024),
            nn.ReLU(True),
            nn.Linear(1024, channels*img_size*img_size),
            nn.Tanh()
        )

    def forward(self, z):
        img = self.model(z)
        img = img.view(z.size(0), channels, img_size, img_size)
        return img

In [26]:
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(channels*img_size*img_size, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, img):
        img_flat = img.view(img.size(0), -1)
        validity = self.model(img_flat)
        return validity

In [27]:
generator = Generator().to(device)
discriminator = Discriminator().to(device)

optimizer_G = optim.Adam(generator.parameters(), lr=lr, betas=(0.5, 0.999))
optimizer_D = optim.Adam(discriminator.parameters(), lr=lr, betas=(0.5, 0.999))
adversarial_loss = nn.BCELoss()

In [28]:
for epoch in range(epochs):
    for i, imgs in enumerate(dataloader):
        real_imgs = imgs.to(device)
        batch_size_curr = real_imgs.size(0)

        # Labels
        valid = torch.ones(batch_size_curr, 1).to(device)
        fake = torch.zeros(batch_size_curr, 1).to(device)

        # -----------------
        #  Train Generator
        # -----------------
        optimizer_G.zero_grad()
        z = torch.randn(batch_size_curr, latent_dim).to(device)
        gen_imgs = generator(z)
        g_loss = adversarial_loss(discriminator(gen_imgs), valid)
        g_loss.backward()
        optimizer_G.step()

        # ---------------------
        #  Train Discriminator
        # ---------------------
        optimizer_D.zero_grad()
        real_loss = adversarial_loss(discriminator(real_imgs), valid)
        fake_loss = adversarial_loss(discriminator(gen_imgs.detach()), fake)
        d_loss = (real_loss + fake_loss) / 2
        d_loss.backward()
        optimizer_D.step()

        if i % 50 == 0:
            print(f"[Epoch {epoch+1}/{epochs}] [Batch {i}/{len(dataloader)}] "
                  f"[D loss: {d_loss.item():.4f}] [G loss: {g_loss.item():.4f}]")

    # ---------------------
    #  Save Checkpoints
    # ---------------------
    # Always overwrite latest
    torch.save(generator.state_dict(), f"{model_dir}/generator_latest.pth")
    torch.save(discriminator.state_dict(), f"{model_dir}/discriminator_latest.pth")

    # Optionally keep one every 10 epochs
    if (epoch + 1) % 10 == 0:
        torch.save(generator.state_dict(), f"{model_dir}/generator_epoch_{epoch+1}.pth")
        torch.save(discriminator.state_dict(), f"{model_dir}/discriminator_epoch_{epoch+1}.pth")

    # ---------------------
    #  Save Sample Images
    # ---------------------
    with torch.no_grad():
        sample_z = torch.randn(16, latent_dim).to(device)
        sample_imgs = generator(sample_z)
        sample_imgs = (sample_imgs + 1) / 2  # scale back to [0,1]
        grid = utils.make_grid(sample_imgs, nrow=4)
        utils.save_image(grid, f"{output_dir}/epoch_{epoch+1}.png")


[Epoch 1/80] [Batch 0/255] [D loss: 0.6919] [G loss: 0.6916]
[Epoch 1/80] [Batch 50/255] [D loss: 1.7658] [G loss: 0.0310]
[Epoch 1/80] [Batch 100/255] [D loss: 0.3649] [G loss: 0.8382]
[Epoch 1/80] [Batch 150/255] [D loss: 0.4778] [G loss: 0.5853]
[Epoch 1/80] [Batch 200/255] [D loss: 0.7241] [G loss: 0.2759]
[Epoch 1/80] [Batch 250/255] [D loss: 0.6046] [G loss: 0.4223]


RuntimeError: [enforce fail at inline_container.cc:424] . unexpected pos 2733120 vs 2733008

In [ ]:
torch.save(generator.state_dict(), f"{model_dir}/generator_final.pth")
torch.save(discriminator.state_dict(), f"{model_dir}/discriminator_final.pth")

In [ ]:
z = torch.randn(16, latent_dim).to(device)
gen_imgs = generator(z)
gen_imgs = (gen_imgs + 1) / 2
grid = utils.make_grid(gen_imgs, nrow=4)
utils.save_image(grid, f"{output_dir}/final_generated.png")